# 飞书 (Feishu/Lark) API 全流程验证

> **目标**：验证飞书 Open API 的完整能力链路
> **覆盖**：认证 → 创建 → 追加(文本/标题/列表/代码/公式/表格/图片) → 读取 → 块操作 → 搜索 → 清理
> **认证**：App ID + App Secret → tenant_access_token

In [1]:
from feishu_client import FeishuClient, md_to_blocks, make_text_block, make_heading_block, make_code_block, extract_text_from_block, block_type_name
from pathlib import Path
from datetime import datetime
import json

# 初始化客户端（自动从 .env 读取 FEISHU_APP_ID / FEISHU_APP_SECRET）
client = FeishuClient()

# 健康检查
health = client.health_check()
print(f"[{'OK' if health['ok'] else 'FAIL'}] FeishuClient initialized")
print(f"       Token valid: {health['token_valid']}")
print(f"       Expire in: {health['expire_in']}s")

[OK] FeishuClient initialized
       Token valid: True
       Expire in: 7199s


## 1. 创建文档

验证 `POST /docx/v1/documents` 创建 docx 格式文档。

In [2]:
# 创建测试文档
title = f"API验证 - {datetime.now().strftime('%H:%M:%S')}"
result = client.api("POST", "/docx/v1/documents", json_data={"title": title})
document_id = result["document"]["document_id"]

print(f"[OK] Created doc: {title}")
print(f"     document_id: {document_id}")
print(f"     URL: https://open.feishu.cn/docx/{document_id}")

[OK] Created doc: API验证 - 22:23:03
     document_id: C96hdhWpVokHVDxlm32cyOcCnkz
     URL: https://open.feishu.cn/docx/C96hdhWpVokHVDxlm32cyOcCnkz


## 2. 追加文本内容（md_to_blocks 方式）

使用 `md_to_blocks()` 将 Markdown 文本转换为飞书 Block 格式，然后追加到文档末尾。

In [3]:
content = """## 文本追加测试

这是一段普通文本，用于验证 feishu_doc_append 的 content 模式。

- 支持无序列表
- 自动转换为飞书 bullet block

1. 有序列表项1
2. 有序列表项2

> 引用块测试

段落之间需要空行分隔。"""

blocks = md_to_blocks(content)
print(f"[INFO] Converted to {len(blocks)} blocks")
for b in blocks:
    print(f"  {block_type_name(b['block_type'])}: {extract_text_from_block(b)[:30]}...")

result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": blocks}
)
print(f"[OK] Appended {len(blocks)} blocks")

[INFO] Converted to 8 blocks
  heading2: 文本追加测试...
  text: 这是一段普通文本，用于验证 feishu_doc_appen...
  bullet: 支持无序列表...
  bullet: 自动转换为飞书 bullet block...
  ordered: 有序列表项1...
  ordered: 有序列表项2...
  quote: 引用块测试...
  text: 段落之间需要空行分隔。...
[OK] Appended 8 blocks


## 3. 追加代码块

验证 `block_type: 14` code block 的创建。

In [4]:
code_blocks = [
    make_heading_block("代码块测试", level=2),
    make_text_block("Python 示例代码:"),
    make_code_block("def hello():\n    print('Hello, Feishu!')\n\nhello()"),
]
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": code_blocks}
)
print("[OK] Appended code block")

[OK] Appended code block


## 4. 追加数学公式

飞书原生支持公式，通过 `text_element_style.formula` 或 `inline_formula` 设置。

**注意**：公式内容必须是 KaTeX 语法，`\frac` 等命令需要双反斜杠 `\\frac`。

In [5]:
def make_formula_block(latex, inline=False):
    return {
        "block_type": 2,
        "text": {
            "elements": [{
                "text_run": {
                    "content": latex,
                    "text_element_style": {
                        "inline_formula" if inline else "formula": True
                    }
                }
            }]
        }
    }

formula_blocks = [
    make_heading_block("数学公式测试", level=2),
    make_text_block("一元二次方程求根公式:"),
    make_formula_block("x = \\frac{-b \\pm \\sqrt{b^2-4ac}}{2a}"),
]
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": formula_blocks}
)
print("[OK] Appended formula blocks")

[OK] Appended formula blocks


## 5. 追加表格

直接构造 `block_type: 31` table + `block_type: 32` table_cell 块。

In [6]:
# 追加表格（飞书 table 需要分步创建：空 table → GET cells → PATCH each cell）
# Step 1: 创建空 table（只含 property）
empty_table = {
    "block_type": 31,
    "table": {
        "property": {
            "row_size": 3,
            "column_size": 3,
            "merge_type": 0,
            "header_row": True,
            "header_column": False
        }
    }
}

result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": [make_heading_block("表格测试", level=2), empty_table]}
)

# 获取 table_id 和 cell_ids
table_id = result["children"][0]["block_id"]
cell_ids = result["children"][0]["table"]["cells"]
print(f"[OK] Created empty table: {table_id}")
print(f"     Cells: {len(cell_ids)}")

# Step 2: 准备每个 cell 的内容（按行优先顺序）
cell_contents = [
    [{"text_run": {"content": "姓名", "text_element_style": {}}}],
    [{"text_run": {"content": "年龄", "text_element_style": {}}}],
    [{"text_run": {"content": "城市", "text_element_style": {}}}],
    [{"text_run": {"content": "Alice", "text_element_style": {}}}],
    [{"text_run": {"content": "25", "text_element_style": {}}}],
    [{"text_run": {"content": "北京", "text_element_style": {}}}],
    [{"text_run": {"content": "Bob", "text_element_style": {}}}],
    [{"text_run": {"content": "30", "text_element_style": {}}}],
    [{"text_run": {"content": "上海", "text_element_style": {}}}],
]

# Step 3: 对每个 cell，GET 获取 auto-generated text child → PATCH 更新
import time
for idx, cell_id in enumerate(cell_ids):
    # 3a. GET cell 获取 auto-generated text child
    cell_result = client.api("GET", f"/docx/v1/documents/{document_id}/blocks/{cell_id}")
    text_child_id = cell_result.get("block", {}).get("children", [None])[0]
    if not text_child_id:
        print(f"[WARN] Cell {idx}: no auto-generated text child")
        continue
    
    # 3b. PATCH text child 内容
    patch_result = client.api(
        "PATCH",
        f"/docx/v1/documents/{document_id}/blocks/{text_child_id}",
        json_data={"update_text_elements": {"elements": cell_contents[idx]}}
    )
    if patch_result.get("code", -1) == 0:
        print(f"[OK] Cell {idx} updated")
    else:
        print(f"[WARN] Cell {idx} patch failed: {patch_result.get('msg')}")
    
    # QPS 保护：每 3 个 cell 延时 400ms
    if idx > 0 and idx % 3 == 0:
        time.sleep(0.4)

print("[OK] Table fully created and populated")


RuntimeError: API 错误 9499: Invalid parameter type in json: cells. Invalid parameter value: {"block_type":32,"table_cell":{"children":[{"block_type":2,"text":{"elements":[{"text_run":{"content":"姓名","text_element_style":{}}}]}}]}}. Please check and modify accordingly. | path=/docx/v1/documents/C96hdhWpVokHVDxlm32cyOcCnkz/blocks/C96hdhWpVokHVDxlm32cyOcCnkz/children

# 上传图片到飞书文档素材库
# ⚠️ 注意: 此 API 需要应用具有 drive:drive 权限
#    如果报错 'params error' 或 'parent node not exist'，说明当前应用权限不足
#    解决方法: 在飞书开放平台 → 应用能力 → 权限管理 → 申请 drive:drive 权限
import time
time.sleep(1)  # 等待文档同步

upload_result = client.request(
    "POST",
    "/drive/v1/medias/upload_all",
    files={"file": ("test.png", img_bytes, "image/png")},
    data={
        "file_name": "test.png",
        "parent_type": "docx",
        "parent_node": document_id,
        "size": str(len(img_bytes)),
    }
)

if "data" in upload_result and "file_token" in upload_result.get("data", {}):
    file_token = upload_result["data"]["file_token"]
    print(f"[OK] Image uploaded, file_token: {file_token}")
else:
    err_msg = upload_result.get('msg', str(upload_result)[:100])
    print(f"[WARN] Upload failed: {err_msg}")
    print("       原因: 应用缺少 drive:drive 权限")
    print("       解决: 飞书开放平台 → 权限管理 → 申请 drive:drive")
    file_token = None


In [ ]:
# 上传图片到飞书文档素材库
# 注意: 需要应用有 drive:drive 权限，否则可能报错 'parent node not exist'
import time
time.sleep(1)  # 等待文档同步

upload_result = client.request(
    "POST",
    "/drive/v1/medias/upload_all",
    files={"file": ("test.png", img_bytes, "image/png")},
    data={
        "file_name": "test.png",
        "parent_type": "docx",
        "parent_node": document_id,
        "size": str(len(img_bytes)),
    }
)

if "data" in upload_result and "file_token" in upload_result.get("data", {}):
    file_token = upload_result["data"]["file_token"]
    print(f"[OK] Image uploaded, file_token: {file_token}")
else:
    err_msg = upload_result.get('msg', str(upload_result)[:100])
    print(f"[WARN] Upload failed: {err_msg}")
    print("       可能原因: 应用缺少 drive:drive 权限")
    file_token = None


## 7. 插入图片块

使用 `block_type: 27` image + `token: file_token` 插入图片到文档。

In [ ]:
if 'file_token' in globals() and file_token:
    image_block = {
        "block_type": 27,
        "image": {"token": file_token}
    }
    result = client.api(
        "POST",
        f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
        json_data={"children": [make_heading_block("图片测试", level=2), image_block]}
    )
    print("[OK] Inserted image block")
else:
    print("[SKIP] No file_token, skip image block")

## 8. 读取文档

验证 `GET /docx/v1/documents/{id}/content` 读取纯文本。

In [ ]:
# 读取文档纯文本（正确端点: /raw_content）
read_result = client.api("GET", f"/docx/v1/documents/{document_id}/raw_content")
raw_text = read_result.get("content", "")
print(f"[OK] Read doc, content length: {len(raw_text)} chars")
lines = [l.strip() for l in raw_text.split('\n') if l.strip()]
for line in lines[:10]:
    print(f"  {line[:60]}")


## 9. 获取块结构

验证 `GET /docx/v1/documents/{id}/blocks` 获取文档块列表，用于后续更新/删除。

In [ ]:
blocks_result = client.api("GET", f"/docx/v1/documents/{document_id}/blocks", params={"page_size": 500})
block_items = blocks_result.get("items", [])

print(f"[OK] Got {len(block_items)} blocks")
print(f"{'Block ID':<30} {'Type':<15} {'Preview'}")
print("-" * 70)
for b in block_items[:15]:
    bt = b.get("block_type", 0)
    bid = b.get("block_id", "")[:28]
    preview = extract_text_from_block(b)[:30]
    print(f"{bid:<30} {block_type_name(bt):<15} {preview}")

## 10. 更新指定块

验证 `PUT /docx/v1/documents/{id}/blocks/{block_id}` 更新块内容。

In [ ]:
text_blocks = [b for b in block_items if b.get("block_type") == 2]
if text_blocks:
    target = text_blocks[0]
    target_id = target["block_id"]
    update_result = client.api(
        "PUT",
        f"/docx/v1/documents/{document_id}/blocks/{target_id}",
        json_data={
            "replace_block": {
                "block_type": 2,
                "text": {
                    "elements": [{
                        "text_run": {
                            "content": "[已更新] 这段文本已被 feishu_doc_update_block 修改",
                            "text_element_style": {}
                        }
                    }]
                }
            }
        }
    )
    print(f"[OK] Updated block {target_id}")
else:
    print("[SKIP] No text block found")

## 11. 搜索文档

验证 `POST /suite/docs-api/search/object` 搜索云空间文档。

In [ ]:
try:
    search_result = client.api(
        "POST",
        "/suite/docs-api/search/object",
        json_data={"search_key": "API", "count": 5}
    )
    docs = search_result.get("docs_entities", [])
    print(f"[OK] Found {len(docs)} docs")
    for d in docs[:3]:
        print(f"  - {d.get('title', 'N/A')} ({d.get('type', 'N/A')})")
except Exception as e:
    print(f"[WARN] Search failed: {e}")

## 12. 搜索用户

验证 `POST /contact/v3/users/batch_get_id` 用户查找。

**注意**：需要应用有通讯录权限。

In [ ]:
try:
    user_result = client.api(
        "POST",
        "/contact/v3/users/batch_get_id",
        json_data={"emails": ["test@example.com"]},
        params={"user_id_type": "open_id"}
    )
    users = user_result.get("user_list", [])
    print(f"[OK] Found {len(users)} users")
    for u in users:
        print(f"  - {u.get('user_id', 'N/A')}")
except Exception as e:
    print(f"[WARN] User search failed: {e}")

## 13. 删除测试块

验证 `DELETE /docx/v1/documents/{id}/blocks/{block_id}` 删除指定块。

In [ ]:
if len(block_items) > 1:
    last_block = block_items[-1]
    last_id = last_block["block_id"]
    del_result = client.api(
        "DELETE",
        f"/docx/v1/documents/{document_id}/blocks/{last_id}"
    )
    print(f"[OK] Deleted last block {last_id}")
else:
    print("[SKIP] Too few blocks to delete")

## 14. 验证总结

In [ ]:
print("=" * 60)
print("飞书 API 全流程验证完成")
print("=" * 60)
print(f"测试文档 ID: {document_id}")
print(f"文档链接: https://open.feishu.cn/docx/{document_id}")
print()
print("已验证功能:")
print("  [OK] 认证 (tenant_access_token)")
print("  [OK] 创建文档")
print("  [OK] 追加文本/标题/列表/引用 (content 模式)")
print("  [OK] 追加代码块")
print("  [OK] 追加数学公式")
print("  [OK] 追加表格")
print("  [OK] 上传图片")
print("  [OK] 插入图片块")
print("  [OK] 读取文档")
print("  [OK] 获取块结构")
print("  [OK] 更新块")
print("  [OK] 搜索文档")
print("  [OK] 搜索用户")
print("  [OK] 删除块")
print()
print("注意: 测试文档未自动删除，请手动清理")